# KrishiBazar AI - Time-Series Feature Engineering

This notebook reproduces the leakage-safe base features in `src/features/engineering.py`. Features are calculated independently for every commodity and market series.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_orissa_mandi.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_ready_features.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Expected cleaned data at {INPUT_PATH}.')

In [ ]:
df = pd.read_csv(INPUT_PATH)
df['price date'] = pd.to_datetime(df['price date'], errors='coerce')
df = df.dropna(subset=['commodity', 'market name', 'price date', 'modal_price']).copy()
df = df.sort_values(['commodity', 'market name', 'price date']).reset_index(drop=True)
print(f'Input rows: {len(df):,}')
df[['commodity', 'market name', 'price date', 'modal_price']].head()

## Calendar and seasonality features

Season mapping follows the training pipeline: Kharif/monsoon (June-September), Rabi/winter (October-February), and Zaid/summer (March-May).

In [ ]:
df['month'] = df['price date'].dt.month
df['day_of_week'] = df['price date'].dt.dayofweek
df['quarter'] = df['price date'].dt.quarter

def season_for_month(month):
    if month in (6, 7, 8, 9):
        return 1
    if month in (10, 11, 12, 1, 2):
        return 2
    return 3

df['season'] = df['month'].map(season_for_month)
df[['price date', 'month', 'day_of_week', 'quarter', 'season']].head()

## Lag and rolling features

Each rolling calculation is shifted by one observation, preventing the model from using today's modal price to predict today's modal price.

In [ ]:
series_keys = ['commodity', 'market name']
grouped = df.groupby(series_keys, observed=True)['modal_price']

df['lag_1'] = grouped.shift(1)
df['lag_3'] = grouped.shift(3)
df['lag_7'] = grouped.shift(7)
df['rolling_mean_3'] = grouped.transform(lambda values: values.shift(1).rolling(3).mean())
df['rolling_mean_7'] = grouped.transform(lambda values: values.shift(1).rolling(7).mean())
df['rolling_std_7'] = grouped.transform(lambda values: values.shift(1).rolling(7).std())

base_features = ['month', 'day_of_week', 'quarter', 'season', 'lag_1', 'lag_3', 'lag_7',
                 'rolling_mean_3', 'rolling_mean_7', 'rolling_std_7']
df[series_keys + ['price date', 'modal_price'] + base_features].head(12)

In [ ]:
ml_ready = df.dropna(subset=['lag_1', 'rolling_mean_7']).copy()
summary = (ml_ready.groupby('commodity', observed=True)
                   .agg(rows=('modal_price', 'size'), markets=('market name', 'nunique'))
                   .sort_values('rows', ascending=False))
print(f'ML-ready rows: {len(ml_ready):,} ({len(ml_ready) / len(df):.1%} of cleaned rows)')
summary

## Advanced training features

The training module adds lag-only momentum, volatility, previous price-spread, and categorical features. Importing it here keeps notebook exploration aligned with production training.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.models.train import add_derived_features

training_frame = add_derived_features(ml_ready)
advanced_columns = ['days_since_last', 'prev_spread', 'mom_1_3', 'mom_1_7',
                    'dev_rm7', 'vol_pct', 'rm3_over_rm7']
training_frame[advanced_columns].describe().T

In [ ]:
# Save the same artifact consumed by src/models/train.py.
ml_ready.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(ml_ready):,} rows to {OUTPUT_PATH}')

## Next step

Train and evaluate the per-crop forecasting models with `python -m src.models.train`. It performs a chronological holdout and compares the model with the carry-forward `lag_1` baseline.